# Clusterização K-Means++ (As Tribos Literárias)

Neste notebook, aplicamos o algoritmo de aprendizado não supervisionado **K-Means++** para agrupar os livros em grandes "Tribos Literárias".

Para contornar a Maldição da Dimensionalidade causada pelos mais de 1.100 gêneros literários, utilizamos a redução de dimensionalidade via **SVD (Truncated Singular Value Decomposition)** antes de adicionar as características numéricas (`rating`, `pages` e `totalratings`).

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 1. Engenharia de Features e Pré-processamento

In [2]:
df = pd.read_parquet('../../Machine Learning/data/processed/books_clean.parquet')
df = df.dropna(subset=['genre_list', 'rating', 'pages', 'totalratings']).copy()

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genre_list'])

cols_numericas = ['rating', 'pages', 'totalratings']
scaler = MinMaxScaler()
numeric_matrix = scaler.fit_transform(df[cols_numericas])

## 2. Busca de Hiperparâmetros: SVD vs Qualidade do Cluster
Testamos diferentes valores de `n_components` para o SVD, variando também o número de clusters $K$, para descobrir qual configuração alcança a maior coesão (Silhouette Score).

In [3]:
n_components_list = [2, 3, 5, 7, 10, 15, 20]
K_range = range(2, 9)
melhores_silhouettes = []

sample_size = min(10000, genre_matrix.shape[0])
np.random.seed(42)
sample_indices = np.random.choice(genre_matrix.shape[0], sample_size, replace=False)

print("Aguarde, rodando Grid Search...")
for n_comp in n_components_list:
    svd = TruncatedSVD(n_components=n_comp, random_state=42)
    genre_svd = svd.fit_transform(genre_matrix)
    X_cluster = np.hstack((genre_svd, numeric_matrix))
    X_sample = X_cluster[sample_indices]
    
    max_sil = -1
    for k in K_range:
        kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
        kmeans.fit(X_cluster)
        score = silhouette_score(X_sample, kmeans.labels_[sample_indices])
        if score > max_sil:
            max_sil = score
            
    melhores_silhouettes.append(max_sil)
    print(f"SVD n_components={n_comp} -> Melhor Silhouette Score = {max_sil:.4f}")

Aguarde, rodando Grid Search...
SVD n_components=2 -> Melhor Silhouette Score = 0.6015
SVD n_components=3 -> Melhor Silhouette Score = 0.5215
SVD n_components=5 -> Melhor Silhouette Score = 0.4494
SVD n_components=7 -> Melhor Silhouette Score = 0.4023
SVD n_components=10 -> Melhor Silhouette Score = 0.3035
SVD n_components=15 -> Melhor Silhouette Score = 0.2351
SVD n_components=20 -> Melhor Silhouette Score = 0.2228


In [4]:
fig = px.line(
    x=[str(n) for n in n_components_list], 
    y=melhores_silhouettes, 
    markers=True,
    title='Impacto da Compressão do SVD na Qualidade das Tribos',
    labels={'x': 'Número de Componentes do SVD', 'y': 'Melhor Silhouette Score Obtido'}
)
fig.update_traces(line_color='purple', marker=dict(size=10))
fig.show()

## 3. Seleção do K com SVD (n_components = 2)
Conforme evidenciado pelo gráfico acima, a configuração com **`n_components = 2`** no SVD produziu os agrupamentos mais fortes matematicamente. Agora, vamos isolar esse cenário e plotar os gráficos de Inércia (Elbow) e Silhouette para descobrir qual foi o exato valor de $K$ que atingiu esse pico.

In [5]:
svd_final = TruncatedSVD(n_components=2, random_state=42)
genre_svd_final = svd_final.fit_transform(genre_matrix)
X_final = np.hstack((genre_svd_final, numeric_matrix))
X_sample_final = X_final[sample_indices]

inertias = []
silhouettes = []

print("Avaliando valores de K para n_components=2...")
for k in K_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_final)
    inertias.append(kmeans.inertia_)
    score = silhouette_score(X_sample_final, kmeans.labels_[sample_indices])
    silhouettes.append(score)
    print(f"K={k} -> Inércia: {kmeans.inertia_:.0f} | Silhouette: {score:.4f}")

Avaliando valores de K para n_components=2...
K=2 -> Inércia: 23180 | Silhouette: 0.5650
K=3 -> Inércia: 9677 | Silhouette: 0.6015
K=4 -> Inércia: 7082 | Silhouette: 0.5279
K=5 -> Inércia: 5055 | Silhouette: 0.4880
K=6 -> Inércia: 4297 | Silhouette: 0.4441
K=7 -> Inércia: 3773 | Silhouette: 0.4559
K=8 -> Inércia: 3313 | Silhouette: 0.4207


In [6]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=list(K_range), y=inertias, name="Inércia", mode='lines+markers', line=dict(color='blue')),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(x=list(K_range), y=silhouettes, name="Silhouette Score", mode='lines+markers', line=dict(color='red', dash='dash')),
    secondary_y=True,
)

fig.update_layout(
    title_text="Método do Cotovelo e Silhouette Score (SVD=2)"
)
fig.update_xaxes(title_text="Número de Clusters (K)", tickmode='linear', tick0=2, dtick=1)
fig.update_yaxes(title_text="<b>Inércia</b>", secondary_y=False)
fig.update_yaxes(title_text="<b>Silhouette Score</b>", secondary_y=True)

fig.show()

## 4. O Modelo Final e o Data Storytelling (K=3)
Com base na análise matemática (Silhouette Score de 0.6015), o número ideal de Tribos Literárias para o nosso catálogo é **K=3**. 

Abaixo, realizamos o treinamento oficial do modelo e extraímos a "personalidade" de cada uma dessas 3 Tribos para apresentarmos no Dashboard.

In [7]:
# Treinamento Oficial do K-Means com K=3
kmeans_final = KMeans(n_clusters=3, init='k-means++', random_state=42, n_init=10)
clusters = kmeans_final.fit_predict(X_final)

# Anexando os resultados ao DataFrame original (apenas os válidos que não eram nulos)
df_final = df.copy()
df_final['Cluster'] = clusters

print("Modelo treinado e tribos designadas com sucesso!")

Modelo treinado e tribos designadas com sucesso!


### Extraindo as Personalidades das Tribos
Vamos agrupar os dados por Cluster e ver a média de páginas, notas e avaliações, além dos gêneros predominantes.

In [8]:
from collections import Counter

for i in range(3):
    cluster_data = df_final[df_final['Cluster'] == i]
    
    media_paginas = cluster_data['pages'].mean()
    media_nota = cluster_data['rating'].mean()
    media_avals = cluster_data['totalratings'].mean()
    
    # Pegando todos os gêneros do cluster para ver quais mais aparecem
    todos_generos = [g for sublist in cluster_data['genre_list'] for g in sublist]
    top_3_generos = [g[0] for g in Counter(todos_generos).most_common(3)]
    
    print(f"\n{'='*40}")
    print(f"Tiragem da Tribo {i} ({len(cluster_data)} livros)")
    print(f"{'='*40}")
    print(f"- Páginas Médias: {media_paginas:.0f}")
    print(f"- Nota Média: {media_nota:.2f}")
    print(f"- Avaliações Médias: {media_avals:.0f}")
    print(f"- Top 3 Gêneros: {', '.join(top_3_generos)}")


Tiragem da Tribo 0 (33796 livros)
- Páginas Médias: 262
- Nota Média: 3.86
- Avaliações Médias: 7498
- Top 3 Gêneros: romance, fiction, fantasy

Tiragem da Tribo 1 (21665 livros)
- Páginas Médias: 232
- Nota Média: 3.86
- Avaliações Médias: 90
- Top 3 Gêneros: romance, childrens, sequential art

Tiragem da Tribo 2 (28593 livros)
- Páginas Médias: 295
- Nota Média: 3.95
- Avaliações Médias: 1458
- Top 3 Gêneros: nonfiction, history, religion


### Visualização 2D das Tribos
Como usamos SVD com 2 componentes, nós já temos as coordenadas X e Y perfeitas para desenhar o nosso mapa literário!

In [9]:
# Preparando os dados para o Plotly
df_plot = pd.DataFrame({
    'Componente Principal 1 (SVD)': genre_svd_final[:, 0],
    'Componente Principal 2 (SVD)': genre_svd_final[:, 1],
    'Cluster': df_final['Cluster'].astype(str),
    'Título': df_final['title']
})

# Para o gráfico não travar o navegador, pegamos uma amostra aleatória de 3.000 livros
df_plot_sample = df_plot.sample(3000, random_state=42)

fig = px.scatter(
    df_plot_sample, 
    x='Componente Principal 1 (SVD)', 
    y='Componente Principal 2 (SVD)', 
    color='Cluster', 
    hover_name='Título',
    title='Mapa Interativo das 3 Tribos Literárias (Amostra de 3000 livros)',
    category_orders={'Cluster': ['0', '1', '2']},
    color_discrete_sequence=px.colors.qualitative.Set1
)

fig.update_traces(marker=dict(size=6, opacity=0.7, line=dict(width=0.5, color='DarkSlateGrey')))
fig.show()

## 5. Exportação para o Dashboard
Por fim, salvamos o resultado.

In [ ]:
# Salvando apenas as colunas essenciais para o Dashboard não ficar pesado
df_export = df_final[['isbn', 'title', 'author', 'rating', 'pages', 'Cluster']].copy()
df_export['svd_x'] = genre_svd_final[:, 0]
df_export['svd_y'] = genre_svd_final[:, 1]
df_export.to_parquet('../data/processed/livros_com_clusters.parquet', index=False)
print("Arquivo 'livros_com_clusters.csv' exportado com sucesso para o Dashboard!")

Arquivo 'livros_com_clusters.csv' exportado com sucesso para o Dashboard!
